# XRD Pipeline — Phase Identification + Enhanced Rietveld

**Stage 1** — DARA tree search to find the best-matching phase combination per pattern.

**Stage 2** — Enhanced BGMN Rietveld on the best solution: adds preferred orientation (SPHAR2),
Lorentzian size broadening (b1), and Gaussian broadening (k1) to push Rwp below 10%.

> Note: `GSASIIscriptable` is installed but `pypowder` (profile calc binary) is missing on this machine,
> so Stage 2 uses DARA/BGMN with extended phase params — same physics, same engine as Stage 1.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
from pathlib import Path

DATA_DIR        = Path("phase search\\all_xy_files")           # .xy files to process
CIFS_DIR        = Path("phase search/cifs")  # shared CIF cache
CHEMICAL_SYSTEM = "Li-Ni-Mn-Co-O-H"
WAVELENGTH      = "Cu"
INSTRUMENT      = "Aeris-fds-Pixcel1d-Medipix3"
EXPRESS_MODE    = False

# Stage 2 Rietveld: advanced phase parameters
RIETVELD_PHASE_PARAMS = {
    "lattice_range": 0.03,          # ±3% lattice freedom
    "b1": "0.003_0.0^0.01",         # Lorentzian size broadening (~200 nm NMC crystallites)
    "k1": "0_0^1",                   # Gaussian size distribution
    "k2": "fixed",                   # microstrain (fixed; enable '0_0^0.001' if peaks still broad)
    "gewicht": "SPHAR2",             # mild preferred orientation — NMC platelet morphology
}
RIETVELD_ANG_MIN = 10
RIETVELD_ANG_MAX = 80
# ──────────────────────────────────────────────────────────────────────────────

In [2]:
import os, re, time, shutil, tempfile, logging
import pandas as pd
from pymatgen.io.cif import CifParser
from dara import search_phases
from dara.cif import Cif
from dara.refine import do_refinement_no_saving

# Windows fix: BGMN holds a file lock on .dia/.str temp files briefly after
# the subprocess exits — ignore cleanup errors so the kernel doesn't crash.
_OrigTempDir = tempfile.TemporaryDirectory
class _WinTempDir(_OrigTempDir):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("ignore_cleanup_errors", True)
        super().__init__(*args, **kwargs)
tempfile.TemporaryDirectory = _WinTempDir
print("Windows tempdir fix applied.")

DATA_DIR.mkdir(exist_ok=True)
CIFS_DIR.mkdir(exist_ok=True)

Windows tempdir fix applied.


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def strip_xy_header(path: Path) -> Path:
    """Strip non-numeric header lines from a COMMANDER-exported .xy file.
    Writes a *_clean.xy sibling; idempotent (skips if already newer)."""
    clean_path = path.parent / (path.stem + "_clean.xy")
    if clean_path.exists() and clean_path.stat().st_mtime >= path.stat().st_mtime:
        return clean_path
    lines = path.read_text(encoding="utf-8").splitlines()
    data_lines = [l for l in lines if re.match(r"^\s*[\d.\-]", l)]
    clean_path.write_text("\n".join(data_lines), encoding="utf-8")
    return clean_path


def cif_stem_to_bgmn_key(stem: str) -> str:
    """Map CIF filename stem to BGMN phase key (strip special chars)."""
    return re.sub(r"[_()\-]", "", stem)


def get_literature_params_nm(cif_path: Path):
    """Return (a, b, c) in nm from a CIF file. Returns (None, None, None) on failure."""
    try:
        parser = CifParser(str(cif_path), occupancy_tolerance=10.0)
        structure = parser.parse_structures(primitive=False)[0]
        lat = structure.lattice
        return lat.a / 10, lat.b / 10, lat.c / 10
    except Exception as exc:
        return None, None, None


def gewicht_str(gw) -> str:
    """Format gewicht (weight fraction) — handles float or (val, err) tuple."""
    if isinstance(gw, (list, tuple)) and len(gw) == 2:
        return f"{gw[0]*100:.1f} ± {gw[1]*100:.1f} wt%"
    if isinstance(gw, float):
        return f"{gw*100:.1f} wt%"
    return str(gw)


print("Helpers defined.")

Helpers defined.


In [4]:
import ray
import dara.search.tree as _dara_tree
from dara.refine import do_refinement_no_saving as _do_ref

# Windows fix: patch remote_do_refinement_no_saving so the tempdir
# cleanup fix runs INSIDE each Ray worker process.
@ray.remote(num_cpus=1)
def _patched_remote_refinement(
    pattern_path, cif_paths, wavelength, instrument_profile,
    phase_params, refinement_params,
):
    import tempfile, shutil, time
    import dara.refine as _dr

    class _RobustTempDir(tempfile.TemporaryDirectory):
        def __init__(self, *args, **kwargs):
            kwargs.setdefault("ignore_cleanup_errors", True)
            super().__init__(*args, **kwargs)
        def cleanup(self):
            for attempt in range(6):
                try:
                    shutil.rmtree(self.name); return
                except PermissionError:
                    if attempt < 5: time.sleep(0.2 * (attempt + 1))
            shutil.rmtree(self.name, ignore_errors=True)

    tempfile.TemporaryDirectory = _RobustTempDir
    _dr.tempfile.TemporaryDirectory = _RobustTempDir

    from subprocess import TimeoutExpired
    from dara.cif2str import CIF2StrError
    import logging
    logger = logging.getLogger("dara.search.tree")

    if len(cif_paths) == 0:
        return None
    try:
        result = _do_ref(
            pattern_path, cif_paths,
            wavelength=wavelength,
            instrument_profile=instrument_profile,
            phase_params=phase_params,
            refinement_params=refinement_params,
        )
    except (RuntimeError, TimeoutExpired, CIF2StrError, ValueError) as e:
        logger.debug(f"Refinement failed for {cif_paths}: {e}")
        return None
    if result.lst_data.rpb == 100:
        return None
    return result

_dara_tree.remote_do_refinement_no_saving = _patched_remote_refinement

# Init Ray (shutdown stale instance if needed)
if ray.is_initialized():
    ray.shutdown()
ray.init(runtime_env={"working_dir": None})

curated_cifs = list(CIFS_DIR.resolve().glob("*.cif"))
print(f"Ray ready. CIF cache: {len(curated_cifs)} phases.")

2026-10-09 16:14:26,284	INFO worker.py:2024 -- Started a local Ray instance.


Ray ready. CIF cache: 311 phases.


## Stage 1 — Phase Identification (DARA tree search)

In [5]:
xy_files = sorted(DATA_DIR.glob("*.xy"))
xy_files = [f for f in xy_files if not f.stem.endswith("_clean")]

if not xy_files:
    print(f"No .xy files found in {DATA_DIR.resolve()}")
else:
    print(f"Found {len(xy_files)} file(s):")
    for f in xy_files:
        print(f"  {f.name}")

Found 1 file(s):
  LGM50 L 175 Middle cathode 0705026_exported.xy


In [6]:
all_results = {}   # fname -> list[SearchResult]

for xy_file in xy_files:
    print(f"\n{'='*70}\nProcessing: {xy_file.name}\n{'='*70}")
    clean_path = strip_xy_header(xy_file)

    if not ray.is_initialized():
        ray.init(runtime_env={"working_dir": None})

    search_results = search_phases(
        pattern_path=str(clean_path),
        phases=curated_cifs,
        wavelength=WAVELENGTH,
        instrument_profile=INSTRUMENT,
        express_mode=EXPRESS_MODE,
        enable_angular_cut=False,
        phase_params={"lattice_range": 0.03, "b1": "0.003_0.0^0.01"},
        rpb_threshold=0.5,
    )
    all_results[xy_file.name] = search_results

    for i, sr in enumerate(search_results):
        rwp = sr.refinement_result.lst_data.rwp
        top = list(sr.refinement_result.lst_data.phases_results.keys())[0]
        n_phases = len(sr.refinement_result.lst_data.phases_results)
        print(f"  Solution {i}: Rwp={rwp:.2f}%  phases={n_phases}  top={top}")

print(f"\nDone. {len(all_results)} file(s) processed.")


Processing: LGM50 L 175 Middle cathode 0705026_exported.xy
2026-10-09 16:14:35,911 INFO dara.search.tree Detecting peaks in the pattern.
2026-10-09 16:14:40,622 INFO dara.search.tree The intensity threshold is automatically set to 9.59 % of maximum peak intensity.
2026-10-09 16:14:40,622 INFO dara.search.tree Creating the root node.
2026-10-09 16:14:40,622 INFO dara.search.tree Refining all the phases in the dataset.
2026-10-09 16:23:17,033 INFO dara.search.tree The initial value of eps2 is automatically set to 0.000004_-0.05^0.05.
2026-10-09 16:23:17,048 INFO dara.search.tree Finished refining 311 phases, with 55 phases removed.
(_remote_expand_node pid=39188) 2026-10-09 16:23:17,849 INFO dara.search.tree Expanding node 6e4a142f-c3ce-11f1-8652-e073e7b8e1ac with current phases [], Rwp = None


(raylet) Stack (most recent call first):
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py"Stack (most recent call first):
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect, line 627 in job_logging_config
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdown
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 1170 in wrapper
(raylet) ", line 2118 in shutdown
(raylet)   File

(_remote_expand_node pid=39188) 2026-10-09 16:26:28,929 INFO dara.search.tree Expanding node 10756c82-c3d0-11f1-86fc-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Li3Mn0.9Ni2.1O6_166_(cod_7222435)-None.cif'), params={'k1': '0.000000_0.0^0.01', 'b1': '0.004850_0.0^0.01'})], Rwp = 11.76


(raylet)   File Stack (most recent call first): [repeated 2x across cluster]
(raylet)   File "2807 in job_logging_config
(raylet) C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in job_logging_config
(raylet)   File "
(raylet)   File "Stack (most recent call first):
(raylet)  in wrapper
(raylet)   File "wrapper
(raylet) wrapper [repeated 2x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py"", line 1170
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\client_mode_hook.py in wrapper
(raylet) Stack (most recent call first): [repeated 9x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config [repeated 9x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in dis

(_remote_expand_node pid=39976) 2026-10-09 16:28:08,089 INFO dara.search.tree Expanding node 2bd4d2e5-c3d0-11f1-bb51-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Li0.8Mn16.384O32_227_(cod_1514081)-None.cif'), params={'k1': '0.000000_0.0^0.01', 'b1': '0.009060_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/MnO2_194_(cod_9014106)-15.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.010000_0.0^0.01'})], Rwp = 24.54
(_remote_expand_node pid=41416) 2026-10-09 16:26:31,918 INFO dara.search.tree Expanding node 10b2be77-c3d0-11f1-aabe-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Mn1.92O4_225_(cod_4001899)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.009970_0.0^0.01'})], Rwp = 32.13 [repeated 5x across cluster]
(_remote_expand_node pid=43304) 2026-10-09 16:28:14,411 INFO dara.searc

(raylet) Stack (most recent call first): [repeated 7x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config [repeated 7x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect [repeated 7x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdown [repeated 7x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\client_mode_hook.py", line 134 in wrapper [repeated 14x across cluster]
(raylet) Stack (most recent call first): [repeated 8x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config [repeated 8x across cluster]
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\P

(_remote_expand_node pid=48160) 2026-10-09 16:32:51,114 INFO dara.search.tree Expanding node 730c46cc-c3d0-11f1-90d9-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Mn(Ni3O4)2_225_(cod_9013975)-0.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.010000_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Co20.96O32_227_(cod_1528447)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.010000_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Co1.5Ni0.5_194_(cod_1525374)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.006130_0.0^0.01'})], Rwp = 28.81
(_remote_expand_node pid=38440) 2026-10-09 16:28:19,061 INFO dara.search.tree Expanding node 38f6b4e4-c3d0-11f1-bad4-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Mn(Ni3O4)2_225_(cod_90139

(raylet) [2026-10-09 16:33:52,175 E 42236 41892] (raylet.exe) worker_pool.cc:627: Some workers of the worker process(24168) have not registered within the timeout. The process is still alive, probably it's hanging during start.
(raylet) [2026-10-09 16:33:57,608 E 42236 41892] (raylet.exe) worker_pool.cc:627: Some workers of the worker process(9744) have not registered within the timeout. The process is still alive, probably it's hanging during start. [repeated 5x across cluster]
(raylet) [2026-10-09 16:34:04,267 E 42236 41892] (raylet.exe) worker_pool.cc:627: Some workers of the worker process(6624) have not registered within the timeout. The process is still alive, probably it's hanging during start. [repeated 9x across cluster]
(raylet) [2026-10-09 16:34:09,307 E 42236 41892] (raylet.exe) worker_pool.cc:627: Some workers of the worker process(46984) have not registered within the timeout. The process is still alive, probably it's hanging during start. [repeated 4x across cluster]
(ra

(_remote_expand_node pid=39024) 2026-10-09 16:35:12,021 INFO dara.search.tree Expanding node 45aa77c2-c3d1-11f1-ab03-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Mn1.92O4_225_(cod_4001899)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.009970_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Li1.26Ni2.04O4_166_(cod_1526282)-None.cif'), params={'k1': '0.000000_0.0^0.01', 'b1': '0.003415_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Li_229_(cod_9008542)-5.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.006580_0.0^0.01'})], Rwp = 27.43 [repeated 2x across cluster]
(_remote_expand_node pid=39024) 2026-10-09 16:32:51,230 INFO dara.search.tree Expanding node 7d8ccbd7-c3d0-11f1-b5da-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Mn

(raylet)   File "Windows fatal exception: access violation
(raylet) 
(raylet) C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py"Stack (most recent call first):
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line , line 627 in job_logging_config
(raylet)   File 1170 in "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in wrapper
(raylet) disconnect


(_remote_expand_node pid=50080) 2026-10-09 16:35:29,870 INFO dara.search.tree Expanding node 53ed03d0-c3d1-11f1-885a-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Mn1.92O4_225_(cod_4001899)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.009970_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Li1.26Ni2.04O4_166_(cod_1526282)-None.cif'), params={'k1': '0.000000_0.0^0.01', 'b1': '0.003415_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Li_229_(cod_9008542)-5.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.006580_0.0^0.01'}), RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Co1.5Ni0.5_194_(cod_1525374)-None.cif'), params={'k1': '0.010000_0.0^0.01', 'b1': '0.006130_0.0^0.01'})], Rwp = 26.66
(_remote_expand_node pid=50080) 2026-10-09 16:35:12,440 INFO dara.search.tree

In [7]:
# Stage 1 summary table — best solution (lowest Rwp) per file
rows = []
for fname, results in all_results.items():
    if not results:
        rows.append({"file": fname, "n_phases": 0, "rwp_phase_id": None,
                     "top_phase": "no result", "a_nm": None, "c_nm": None})
        continue
    best = results[0]
    rwp  = best.refinement_result.lst_data.rwp
    pr_dict = best.refinement_result.lst_data.phases_results
    top_key = list(pr_dict.keys())[0]
    pr = pr_dict[top_key]
    rows.append({
        "file":         fname,
        "n_phases":     len(pr_dict),
        "rwp_phase_id": round(rwp, 2),
        "top_phase":    top_key,
        "a_nm":         round(pr.a[0], 6) if pr.a else None,
        "c_nm":         round(pr.c[0], 6) if pr.c else None,
    })

df_phase_id = pd.DataFrame(rows)
display(df_phase_id)

,file,n_phases,rwp_phase_id,top_phase,a_nm,c_nm
0,LGM50 L 175 Middle cathode 0705026_exported.xy,1,11.76,Li3Mn0.9Ni2.1O6_166_(cod_7222435)-None,0.287592,1.421247


## Stage 2 — Enhanced Rietveld Refinement

Takes the lowest-Rwp phase combination from Stage 1 and re-refines it with:
- **SPHAR2** preferred orientation (layered oxide platelets),
- Lorentzian size broadening **b1** (crystallite size effects),
- Gaussian size distribution **k1**,
- microstrain **k2** held fixed (enable `0_0^0.001` if peaks are still asymmetric).

Expected improvement: ~1–2% Rwp drop vs Stage 1.

In [8]:
rietveld_results = {}   # fname -> RefinementResult

for fname, search_results in all_results.items():
    if not search_results:
        print(f"Skipping {fname}: no phase ID result")
        continue

    best = search_results[0]   # lowest Rwp solution
    rwp_stage1 = best.refinement_result.lst_data.rwp
    cif_paths  = [rp.path for rp in best.phases[0]]

    xy_file    = DATA_DIR / fname
    clean_path = strip_xy_header(xy_file)

    phase_labels = [Path(p).stem for p in cif_paths]
    print(f"\n{'='*70}\n{fname}")
    print(f"  Stage-1 Rwp: {rwp_stage1:.2f}%  |  {len(cif_paths)} phase(s): {phase_labels}")

    rr = do_refinement_no_saving(
        clean_path,
        cif_paths,
        wavelength=WAVELENGTH,
        instrument_profile=INSTRUMENT,
        phase_params=RIETVELD_PHASE_PARAMS,
        refinement_params={"wmin": RIETVELD_ANG_MIN, "wmax": RIETVELD_ANG_MAX},
    )
    rietveld_results[fname] = (best, rr)

    rwp_stage2 = rr.lst_data.rwp
    delta = rwp_stage1 - rwp_stage2
    flag = "✓" if rwp_stage2 < 15 else "⚠ >15%"
    print(f"  Stage-2 Rwp: {rwp_stage2:.2f}%  (Δ = {delta:+.2f}%)  {flag}")

print(f"\nRefinement complete: {len(rietveld_results)} file(s).")


LGM50 L 175 Middle cathode 0705026_exported.xy
  Stage-1 Rwp: 11.76%  |  24 phase(s): ['Li3Mn0.9Ni2.1O6_166_(cod_7222435)-None', 'Li0.65Ni1.08O2_166_(cod_1535512)-None', 'LiNiO2_166_(cod_1533770)-13', 'Li3.03Mn0.3Ni2.67O6_166_(cod_1520787)-None', 'Li3Mn1.5Ni1.497O6_166_(cod_4002582)-None', 'Li3Mn1.2Ni1.8O6_166_(cod_7222436)-None', 'Li2.99995053Mn0.99933044Co0.99933044Ni1.00138858O6_166_(cod_4002443)-None', 'Li2.913Ni3.081O6_166_(cod_1531444)-None', 'Li3.09Mn0.3Co0.3Ni2.31O6_166_(cod_1520789)-None', 'Li3.0Ni3.006O6_166_(cod_1531441)-None', 'Li2.46Ni3.06O6_166_(cod_1526280)-None', 'Li3.6Mn1.5Co0.6Ni0.3O6_166_(cod_1529154)-None', 'Li3Mn0.75Co1.5Ni0.75O6_166_(cod_1529155)-None', 'Li2.88Mn2.37O6_166_(cod_4001904)-None', 'Li0.51Ni1.16O2_166_(cod_1535516)-None', 'Li3.21Mn2.28O6_166_(cod_4001902)-None', 'Li2.88Mn2.22O6_166_(cod_4001905)-None', 'Li2.4Mn2.43O6_166_(cod_4001903)-None', 'Li2.34Mn2.55O6_166_(cod_4001901)-None', 'Li1.26Ni2.04O4_166_(cod_1526282)-None', 'Li2.25Ni3.75O6_166_(cod_1526

In [9]:
# Comparison summary table
rows2 = []
for fname, (best_sr, rr) in rietveld_results.items():
    rwp1 = best_sr.refinement_result.lst_data.rwp
    rwp2 = rr.lst_data.rwp
    pr_dict = rr.lst_data.phases_results
    top_key = list(pr_dict.keys())[0]
    pr = pr_dict[top_key]
    rows2.append({
        "file":           fname,
        "n_phases":       len(pr_dict),
        "Rwp_stage1_%":   round(rwp1, 2),
        "Rwp_stage2_%":   round(rwp2, 2),
        "delta_Rwp":      round(rwp1 - rwp2, 2),
        "top_phase":      top_key,
        "a_nm":           round(pr.a[0], 6) if pr.a else None,
        "c_nm":           round(pr.c[0], 6) if pr.c else None,
    })

df_rietveld = pd.DataFrame(rows2)
display(df_rietveld)

,file,n_phases,Rwp_stage1_%,Rwp_stage2_%,delta_Rwp,top_phase,a_nm,c_nm
0,LGM50 L 175 Middle cathode 0705026_exported.xy,24,11.76,7.29,4.47,Li3Mn0.9Ni2.1O6_166_(cod_7222435)-None,0.288211,1.42209


## Detailed results per file — all phases, lattice params vs literature

In [12]:
for fname, (best_sr, rr) in rietveld_results.items():
    print(f"\n{'='*70}\nFile: {fname}\n{'='*70}")
    print(f"  Stage-1 Rwp: {best_sr.refinement_result.lst_data.rwp:.2f}%  →  "
          f"Stage-2 Rwp: {rr.lst_data.rwp:.2f}%")

    pr_dict   = rr.lst_data.phases_results
    cif_paths = [rp.path for rp in best_sr.phases[0]]
    key_to_path = {cif_stem_to_bgmn_key(Path(p).stem): Path(p) for p in cif_paths}

    for phase_key, pr in pr_dict.items():
        cif_path = key_to_path.get(phase_key)
        lit_a, lit_b, lit_c = get_literature_params_nm(cif_path) if cif_path else (None, None, None)
        label = Path(cif_path).stem if cif_path else phase_key

        print(f"\n  Phase: {label}")
        if pr.gewicht is not None:
            print(f"  Weight fraction: {gewicht_str(pr.gewicht)}")
        print(f"  {'Param':<6} {'Refined (nm)':>14} {'± σ':>10} {'Literature (nm)':>16} {'Δ (%)':>8}")
        print(f"  {'-'*56}")
        for pname, refined_tuple, lit_val in [
            ("a", pr.a, lit_a), ("b", pr.b, lit_b), ("c", pr.c, lit_c)
        ]:
            if refined_tuple is None:
                continue
            if isinstance(refined_tuple, (list, tuple)):
                ref_val, ref_err = refined_tuple
            else:
                ref_val, ref_err = refined_tuple, None
            err_str   = f"± {ref_err:.6f}" if ref_err is not None else ""
            lit_str   = f"{lit_val:.6f}" if lit_val else "N/A"
            delta_str = f"{100*(ref_val-lit_val)/lit_val:+.3f}%" if lit_val else ""
            print(f"  {pname:<6} {ref_val:>14.6f} {err_str:>10} {lit_str:>16} {delta_str:>8}")


File: LGM50 L 175 Middle cathode 0705026_exported.xy
  Stage-1 Rwp: 11.76%  →  Stage-2 Rwp: 7.29%

  Phase: Li3Mn0.9Ni2.1O6_166_(cod_7222435)-None
  Weight fraction: 0.1 wt%
  Param    Refined (nm)        ± σ  Literature (nm)    Δ (%)
  --------------------------------------------------------
  a            0.288211 ± 0.000017              N/A         
  c            1.422090 ± 0.000170              N/A         

  Phase: Li0.65Ni1.08O2_166_(cod_1535512)-None
  Weight fraction: 0.2 wt%
  Param    Refined (nm)        ± σ  Literature (nm)    Δ (%)
  --------------------------------------------------------
  a            0.287474 ± 0.000017              N/A         
  c            1.420170 ± 0.000150              N/A         

  Phase: LiNiO2_166_(cod_1533770)-13
  Weight fraction: 0.0 wt%
  Param    Refined (nm)        ± σ  Literature (nm)    Δ (%)
  --------------------------------------------------------

  Phase: Li3.03Mn0.3Ni2.67O6_166_(cod_1520787)-None
  Weight fraction: 0.0 wt%
 

## Fit visualisation — best Rietveld fit per file

In [11]:
# Shows the observed / calculated / difference plot for each file.
# Call .write_html("fname.html") or .write_image("fname.png") to save.
for fname, (_, rr) in rietveld_results.items():
    print(f"\n{fname}  (Rwp = {rr.lst_data.rwp:.2f}%)")
    fig = rr.visualize()
    fig.show()


LGM50 L 175 Middle cathode 0705026_exported.xy  (Rwp = 7.29%)


## (Optional) Export refined CIF files

In [ ]:
OUT_DIR = Path("refined_cifs")
OUT_DIR.mkdir(exist_ok=True)

for fname, (_, rr) in rietveld_results.items():
    sample = Path(fname).stem.replace(" ", "_")
    for phase_key in rr.lst_data.phases_results:
        try:
            structure = rr.export_structure(phase_key)
            out_path  = OUT_DIR / f"{sample}__{phase_key}_refined.cif"
            structure.to(str(out_path), symprec=1e-3)
            print(f"  Saved: {out_path.name}")
        except Exception as e:
            print(f"  [skip] {fname} / {phase_key}: {e}")